# LCID


In [1]:
import glob
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', None)
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm
import matplotlib.pyplot as plt

# Étude de l'imbalance

In [ ]:
files_csv = glob.glob(os.path.join("/Volumes/T9/CSV_LCID_NASDAQ_PL", "*.csv"))
imb = np.array([])
price = np.array([])
limite = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df = df[:-100]
    imb = np.concatenate([imb, df['imbalance'].to_numpy()])
    price = np.concatenate([price, df['Mean_price_diff'].to_numpy()])

print(f"A quel point c'est long: {len(imb)}")
# ATTENTION L'imbalance est moins l'imbalance et les mid price sont - les midprices, dcp ca chage rien au graphe mais a modif au cas ou

In [ ]:
indices_trie = np.argsort(imb)

bounds = 0.85
imb_trie = imb[indices_trie]
price_trie = price[indices_trie]
mask = (imb_trie >= -bounds)&(imb_trie <= bounds)
imb_trie = imb_trie[mask]
price_trie = price_trie[mask]
group_size = 70000

imb_trie_groups = [imb_trie[i:i + group_size] for i in range(0, len(imb_trie), group_size)]
price_trie_groups = [price_trie[i:i + group_size] for i in range(0, len(price_trie), group_size)]

imb_trie_means = np.array([np.mean(group) for group in imb_trie_groups])
price_trie_means = np.array([np.mean(group) for group in price_trie_groups])

imb_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in price_trie_groups])

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means + 1.96 * imb_trie_std,mode='lines',line=dict(width=0),name='Upper Bound',showlegend=False))
fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means - 1.96 * imb_trie_std,mode='lines',line=dict(width=0),fill='tonexty',fillcolor='blue',name='Intervalle de confiance à 95%',showlegend=True))
fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means,mode='lines',name='Prix',line=dict(color='red'),showlegend=True))
fig.update_layout(title='Imbalance non recentré',xaxis_title='imbalance',yaxis_title='delta_price',showlegend=True)
fig.show()

In [ ]:
nb_bins = 30
counts, bin_edges = np.histogram(imb_trie, bins=nb_bins)
bin_centers = (bin_edges[:-1]+bin_edges[1:])/2

fig = go.Figure()
fig.add_trace(go.Scatter(x=bin_centers,y=counts,mode='lines',name='Density Curve'))
fig.update_layout(title=f"Courbe de distribution de l'imbalance (tous event confondus)",xaxis_title='imbalance',yaxis_title='number of events',showlegend=False)
fig.show()

In [ ]:
imb_trade = np.array([])
price_trade = np.array([])
imb_add = np.array([])
price_add = np.array([])
imb_cancel = np.array([])
price_cancel = np.array([])
limite = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df = df[:-100]
    df_trade = df[df['action'] == 'T']
    df_cancel = df[df['action'] == 'C']
    df_add = df[df['action'] == 'A']
    imb_trade = np.concatenate([imb_trade, df_trade['imbalance'].to_numpy()])
    price_trade = np.concatenate([price_trade, df_trade['Mean_price_diff'].to_numpy()])
    imb_add = np.concatenate([imb_add, df_add['imbalance'].to_numpy()])
    price_add = np.concatenate([price_add, df_add['Mean_price_diff'].to_numpy()])
    imb_cancel = np.concatenate([imb_cancel, df_cancel['imbalance'].to_numpy()])
    price_cancel = np.concatenate([price_cancel, df_cancel['Mean_price_diff'].to_numpy()])
    
imb_tot = [imb_trade,imb_add,imb_cancel]
price_tot = [price_trade,price_add,price_cancel]

print(f"A quel point c'est long: {len(imb_trade)}")

In [ ]:
def visu_imbalance_respec(imb_tot, price_tot, i , string, group_size, bound = 0.95):
    indices_trie = np.argsort(imb_tot[i])
    imb_trie = imb_tot[i][indices_trie]
    price_trie = price_tot[i][indices_trie]
    mask = (imb_trie >= -bound)&(imb_trie <= bound)
    imb_trie = imb_trie[mask]
    price_trie = price_trie[mask]
    imb_trie_groups = [imb_trie[i:i+group_size] for i in range(0, len(imb_trie), group_size)]
    price_trie_groups = [price_trie[i:i+group_size] for i in range(0, len(price_trie), group_size)]
    imb_trie_means = np.array([np.mean(group) for group in imb_trie_groups])
    price_trie_means = np.array([np.mean(group) for group in price_trie_groups])
    imb_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in price_trie_groups])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means + 1.96 * imb_trie_std,mode='lines',line=dict(width=0),name='Upper Bound',showlegend=False))
    fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means - 1.96 * imb_trie_std,mode='lines',line=dict(width=0),fill='tonexty',fillcolor='blue',name='Intervalle de confiance à 95%',showlegend=True))
    fig.add_trace(go.Scatter(x=imb_trie_means,y=price_trie_means,mode='lines',name='Prix',line=dict(color='red'),showlegend=True))
    fig.update_layout(title=f'Imbalance des {string}',xaxis_title='imbalance',yaxis_title='delta_price',showlegend=True)
    fig.show()

    nb_bins = 20
    counts, bin_edges = np.histogram(imb_trie, bins=nb_bins)
    bin_centers = (bin_edges[:-1]+bin_edges[1:])/2

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=bin_centers,y=counts,mode='lines',name='Density Curve'))
    fig.update_layout(title=f"Courbe de distribution de l'imbalance des {string}",xaxis_title='imbalance',yaxis_title='number of events',showlegend=False)
    fig.show()

def visu_event_vs_imbalance(imb_tot, price_tot, string, bound = 0.9):
    fig = go.Figure()
    nb_bins = 25
    counts_tot = []
    bins_centers_tot = []
    for i in range (len(imb_tot)):
        indices_trie = np.argsort(imb_tot[i])
        imb_trie = imb_tot[i][indices_trie]
        price_trie = price_tot[i][indices_trie]
        mask = (imb_trie >= -bound)&(imb_trie <= bound)
        imb_trie = imb_trie[mask]
        price_trie = price_trie[mask]
        counts, bin_edges = np.histogram(imb_trie, bins=nb_bins)
        counts_tot.append(counts)
        bins_centers_tot.append((bin_edges[:-1]+bin_edges[1:])/2)
    counts_total = np.array(counts_tot[0])+np.array(counts_tot[1])+np.array(counts_tot[2])
    for i in range (len(imb_tot)):
        fig.add_trace(go.Scatter(x=bins_centers_tot[i],y=np.array(counts_tot[i])/counts_total,mode='lines',name=f'{string[i]}'))
    fig.update_layout(title=f"Courbes de distribution de l'imbalance (tous les event))",xaxis_title='imbalance',yaxis_title="Probabilité de l'event",showlegend=True)
    fig.show()

In [ ]:
visu_imbalance_respec(imb_tot, price_tot, 0 , 'trades', 5000)
visu_imbalance_respec(imb_tot, price_tot, 1 , 'add', 50000)
visu_imbalance_respec(imb_tot, price_tot, 2 , 'cancel', 40000)
visu_event_vs_imbalance(imb_tot, price_tot, ['Trades', 'Cancel',' Add'], bound = .95)

In [ ]:
imb = np.array([])
intensity = np.array([])
limite = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df = df[:-100]
    imb = np.concatenate([imb, df['imbalance'].to_numpy()])
    intensity = np.concatenate([intensity, df['time_diff'].to_numpy()])

intensity = 1/np.array(intensity)

In [ ]:
indices_trie = np.argsort(imb)

bounds = 0.95
imb_trie = imb[indices_trie]
intensity_trie = intensity[indices_trie]
mask = (imb_trie >= -bounds)&(imb_trie <= bounds)
imb_trie = imb_trie[mask]
intensity_trie = intensity_trie[mask]
group_size = 300000

imb_trie_groups = [imb_trie[i:i + group_size] for i in range(0, len(imb_trie), group_size)]
intensity_trie_groups = [intensity_trie[i:i + group_size] for i in range(0, len(intensity_trie), group_size)]

imb_trie_means = np.array([np.mean(group) for group in imb_trie_groups])
intensity_trie_means = np.array([np.mean(group) for group in intensity_trie_groups])

intensity_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in intensity_trie_groups])

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means + 1.96 * intensity_trie_std,mode='lines',line=dict(width=0),name='Upper Bound',showlegend=False))
fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means - 1.96 * intensity_trie_std,mode='lines',line=dict(width=0),fill='tonexty',fillcolor='blue',name='Intervalle de confiance à 95%',showlegend=True))
fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means,mode='lines',name='intensity',line=dict(color='red'),showlegend=True))
fig.update_layout(title='Intensity (tous events confondus)',xaxis_title='imbalance',yaxis_title='Intensity',showlegend=True)
fig.show()

In [ ]:
imb_trade = np.array([])
intensity_trade = np.array([])
imb_add = np.array([])
intensity_add = np.array([])
imb_cancel = np.array([])
intensity_cancel = np.array([])
limite = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df = df[:-100]
    df_trade = df[df['action'] == 'T']
    df_cancel = df[df['action'] == 'C']
    df_add = df[df['action'] == 'A']
    imb_trade = np.concatenate([imb_trade, df_trade['imbalance'].to_numpy()])
    intensity_trade = np.concatenate([intensity_trade, df_trade['time_diff'].to_numpy()])
    imb_add = np.concatenate([imb_add, df_add['imbalance'].to_numpy()])
    intensity_add = np.concatenate([intensity_add, df_add['time_diff'].to_numpy()])
    imb_cancel = np.concatenate([imb_cancel, df_cancel['imbalance'].to_numpy()])
    intensity_cancel = np.concatenate([intensity_cancel, df_cancel['time_diff'].to_numpy()])
    
imb_tot = [imb_trade,imb_add,imb_cancel]
intensity_tot = [1/np.array(intensity_trade),1/np.array(intensity_add),1/np.array(intensity_cancel)]

print(f"A quel point c'est long: {len(imb_trade)}")

In [ ]:
def visu_intensity_respec(imb_tot, intensity_tot, i , string, group_size, bound = 0.9):
    indices_trie = np.argsort(imb_tot[i])
    imb_trie = imb_tot[i][indices_trie]
    intensity_trie = intensity_tot[i][indices_trie]
    mask = (imb_trie >= -bound) & (imb_trie <= bound)
    imb_trie = imb_trie[mask]
    intensity_trie = intensity_trie[mask]
    imb_trie_groups = [imb_trie[i:i + group_size] for i in range(0, len(imb_trie), group_size)]
    intensity_trie_groups = [intensity_trie[i:i + group_size] for i in range(0, len(intensity_trie), group_size)]
    imb_trie_means = np.array([np.mean(group) for group in imb_trie_groups])
    intensity_trie_means = np.array([np.mean(group) for group in intensity_trie_groups])
    intensity_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in intensity_trie_groups])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means + 1.96 * intensity_trie_std,mode='lines',line=dict(width=0),name='Upper Bound',showlegend=False))
    fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means - 1.96 * intensity_trie_std,mode='lines',line=dict(width=0),fill='tonexty',fillcolor='blue',name='Intervalle de confiance à 95%',showlegend=True))
    fig.add_trace(go.Scatter(x=imb_trie_means,y=intensity_trie_means,mode='lines',name='intensity',line=dict(color='red'),showlegend=True))
    fig.update_layout(title=f"Intensity des {string} en fonction de l'imbalance",xaxis_title='imbalance',yaxis_title='Intensity',showlegend=True)
    fig.show()

In [ ]:
visu_intensity_respec(imb_tot, intensity_tot, 0 , 'trades', 5000)
visu_intensity_respec(imb_tot, intensity_tot, 1 , 'add', 100000)
visu_intensity_respec(imb_tot, intensity_tot, 2 , 'cancel', 90000)

# Calcul Intensités par average event size

In [122]:
def dico_queue_size(sizes, dic):
    for i in range (len(sizes)):
        if sizes[i] not in dic:
            dic[sizes[i]] = [[], [], []]
    return dic

def compute_means(dico):
    sums = 0
    means = 0
    keys = np.array(list(dico.keys()))
    for i in range (len(keys)):
        means = means+keys[i]*len(dico[keys[i]][0])+keys[i]*len(dico[keys[i]][1])+keys[i]*len(dico[keys[i]][2])
        sums = sums+len(dico[keys[i]][0])+len(dico[keys[i]][1])+len(dico[keys[i]][2])
    return means/sums

def filtrage(dico, nombre_bins, threshold=100):
    dico_p = dict(reversed(list(dico.items())))
    keys = list(dico_p.keys())
    

    i = len(keys)
    # while len(dico_p[keys[i]][0])>threshold:
    #     i += 1
    list_ = [keys[j] for j in range (i)]
    nbr_apparition = [len(dico_p[keys[j]][0])+len(dico_p[keys[j]][0])+len(dico_p[keys[j]][0]) for j in range (i)]
    indices = np.argsort(nbr_apparition)

    list_ = np.array(list_)
    list_ = list_[indices]

    #values = np.linspace(0, keys[i], nombre_bins, endpoint=True)
    values = list_[-nombre_bins:]
    keys = np.array(list(dico.keys()))
    
    real_dic = {}
    for i in range (len(keys)):
        real_k_index = np.argmin(np.abs(values-keys[i]))
        real_k = values[real_k_index]
        
        if real_k not in real_dic:
            real_dic[real_k] = [
                np.array(dico[keys[i]][0]),
                np.array(dico[keys[i]][1]),
                np.array(dico[keys[i]][2])
            ]
        else:
            real_dic[real_k] = [
                np.concatenate([real_dic[real_k][0], dico[keys[i]][0]]),
                np.concatenate([real_dic[real_k][1], dico[keys[i]][1]]),
                np.concatenate([real_dic[real_k][2], dico[keys[i]][2]])
            ]
    return real_dic

def remove_nan_from_dico(dico):
    cleaned_dico = {}
    for key, value_lists in dico.items():
        cleaned_value_lists = []
        for value_list in value_lists:
            value_array = np.array(value_list)
            cleaned_array = value_array[~np.isnan(value_array)]
            cleaned_value_lists.append(cleaned_array.tolist())
        cleaned_dico[key] = cleaned_value_lists
    return cleaned_dico

In [91]:
print(np.argsort([1,2,3,4]))

[0 1 2 3]


In [145]:
dic = {}
files_csv = glob.glob(os.path.join("/Volumes/T9/CSV_LCID_NASDAQ_SL", "*.csv"))
average_event_size = []
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df['time_diff'] = df['time_diff']*1.0 # pour le modif en float c'est un timedelta là
    sizes = np.unique(np.array((np.unique(df['imbalance'].to_numpy())).tolist() + (np.unique(df['imbalance'].to_numpy())).tolist()))
    sizes.sort()
    dic = dico_queue_size(sizes, dic)  # Add, Cancel, Trade
    
    for row in df.itertuples():
        average_event_size.append(row.imbalance)
        if row.side == 'A':
            taille = row.imbalance
        elif row.side == 'B':
            taille = row.imbalance
        if row.action == 'A':
            dic[taille][0].append(row.time_diff)
        elif row.action == 'C':
            dic[taille][1].append(row.time_diff)
        elif row.action == 'T':
            dic[taille][2].append(row.time_diff)

100%|██████████| 65/65 [00:03<00:00, 21.06it/s]


In [146]:
taille_add = []
taille_cancel = []
taille_trades = []
add = []
trades = []
cancel = []
mean = 0
leng = 0
intensities = dict(sorted(dic.items()))
intensities = filtrage(intensities, 400, threshold=100)
print('done')


done


In [147]:
print(intensities.keys())
mean = 0
size = 0
for i in intensities:
    mean+=i*(len(intensities[i][0])+len(intensities[i][0])+len(intensities[i][0]))
    size+=(len(intensities[i][0])+len(intensities[i][0])+len(intensities[i][0]))
mean/=size
print(mean)

dict_keys([-0.9246278687708468, -0.9244841576026084, -0.9121315988462916, -0.6473013322824621, -0.6374951095461658, -0.6371735020405166, -0.6130727056019071, -0.6044472462605307, -0.5611848563745785, -0.5303733519530127, -0.5199801900516916, -0.5133541530716883, -0.5113688219868211, -0.5110713350745496, -0.4870125088969427, -0.4865260523582475, -0.4802093555773634, -0.4670641183003098, -0.4589046953844399, -0.4554838709677419, -0.4513693038444411, -0.4427435122644863, -0.4281796566160489, -0.4270553007618542, -0.4179917746796727, -0.4135212350969727, -0.4043715846994535, -0.3936671906579834, -0.3856922474730032, -0.3841153073007998, -0.380321665089877, -0.3731442970339111, -0.3657623132865915, -0.3615587013721269, -0.3603612058268423, -0.3505813892639224, -0.350487962487366, -0.3369211246502668, -0.3306211268271914, -0.3235508478693941, -0.3219882468168462, -0.319591101231075, -0.3172679443818113, -0.3131657020572929, -0.3094439991255556, -0.2946771893409391, -0.2939855053084802, -0.29

In [150]:
taille_add = []
taille_cancel = []
taille_trades = []
add = []
trades = []
cancel = []

for i in intensities:
    if i<100000:
        leng+=1
        tab = np.concatenate((intensities[i][0], intensities[i][1], intensities[i][2]))
        mean+=len(tab)*i
        if len(intensities[i][0])>1:
            taille_add.append(i)
            add.append(1/np.mean(tab)*len(intensities[i][0])/len(tab))
        if len(intensities[i][1])>1:
            taille_cancel.append(i)
            cancel.append(1/np.mean(tab)*len(intensities[i][1])/len(tab))
        if len(intensities[i][2])>1:
            taille_trades.append(i)
            trades.append(1/np.mean(tab)*len(intensities[i][2])/len(tab))

average_sizes = compute_means(intensities)

group_size = 2

indices_add = np.argsort(taille_add)
taille_add = np.array(taille_add)[indices_add]
add = np.array(add)[indices_add]
taille_group = [taille_add[i:i + group_size] for i in range(0, len(taille_add), group_size)]
add_group = [add[i:i + group_size] for i in range(0, len(add), group_size)]
taille_add = np.array([np.mean(group) for group in taille_group])
add = np.array([np.mean(group) for group in add_group])

group_size = 2
indices_add = np.argsort(taille_add)
taille_add = np.array(taille_add)[indices_add]
add = np.array(add)[indices_add]
taille_group = [taille_add[i:i + group_size] for i in range(0, len(taille_add), group_size)]
add_group = [add[i:i + group_size] for i in range(0, len(add), group_size)]
taille_add = np.array([np.mean(group) for group in taille_group])
add = np.array([np.mean(group) for group in add_group])

group_size = 2
indices_cancel = np.argsort(taille_cancel)
taille_cancel = np.array(taille_cancel)[indices_cancel]
cancel = np.array(cancel)[indices_cancel]
taille_group = [taille_cancel[i:i + group_size] for i in range(0, len(taille_cancel), group_size)]
cancel_group = [cancel[i:i + group_size] for i in range(0, len(cancel), group_size)]
taille_cancel = np.array([np.mean(group) for group in taille_group])
cancel = np.array([np.mean(group) for group in cancel_group])

group_size = 2
indices_cancel = np.argsort(taille_cancel)
taille_cancel = np.array(taille_cancel)[indices_cancel]
cancel = np.array(cancel)[indices_cancel]
taille_group = [taille_cancel[i:i + group_size] for i in range(0, len(taille_cancel), group_size)]
cancel_group = [cancel[i:i + group_size] for i in range(0, len(cancel), group_size)]
taille_cancel = np.array([np.mean(group) for group in taille_group])
cancel = np.array([np.mean(group) for group in cancel_group])

indices_trades = np.argsort(taille_trades)
taille_trades = np.array(taille_trades)[indices_trades]
trades = np.array(trades)[indices_trades]
group_size = 3
taille_group = [taille_trades[i:i + group_size] for i in range(0, len(taille_trades), group_size)]
trades_group = [trades[i:i + group_size] for i in range(0, len(trades), group_size)]
taille_trades = np.array([np.mean(group) for group in taille_group])
trades = np.array([np.mean(group) for group in trades_group])

indices_trades = np.argsort(taille_trades)
taille_trades = np.array(taille_trades)[indices_trades]
trades = np.array(trades)[indices_trades]
group_size = 1
taille_group = [taille_trades[i:i + group_size] for i in range(0, len(taille_trades), group_size)]
trades_group = [trades[i:i + group_size] for i in range(0, len(trades), group_size)]
taille_trades = np.array([np.mean(group) for group in taille_group])
trades = np.array([np.mean(group) for group in trades_group])

# taille_add = np.insert(taille_add, 0, 0)
# taille_cancel = np.insert(taille_cancel, 0, 0)
# taille_trades = np.insert(taille_trades, 0, 0)
# add = np.insert(add, 0, 0)
# trades = np.insert(trades, 0, 0)
# cancel = np.insert(cancel, 0, 0)

taille_add_intervalle = 1.96/np.sqrt(60)*0.4
taille_cancel_intervalle = 1.96/np.sqrt(65)*0.4
taille_trades_intervalle = 1.96/np.sqrt(65)*0.1

fig = go.Figure()

fig.add_trace(go.Scatter(

    x=taille_add ,
    y=add + taille_add_intervalle,  # Limite supérieure
    mode='lines',
    line=dict(width=0),  # Ligne invisible pour le remplissage
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=taille_add ,
    y=add - taille_add_intervalle,  # Limite inférieure
    mode='lines',
    fill='tonexty',  # Remplir entre cette courbe et la précédente
    fillcolor='rgba(0, 0, 255, 0.2)',  # Couleur bleue semi-transparente
    line=dict(width=0),
    name='Intervalle Add',
    showlegend=False
))

fig.add_trace(go.Scatter(x=taille_add,y=add,mode='lines',name='Add',showlegend=True,line=dict(color='blue')))

fig.add_trace(go.Scatter(
    x=taille_cancel ,
    y=cancel + taille_cancel_intervalle,  # Limite supérieure
    mode='lines',
    line=dict(width=0),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=taille_cancel ,
    y=cancel - taille_cancel_intervalle,  # Limite inférieure
    mode='lines',
    fill='tonexty',
    fillcolor='rgba(0, 255, 0, 0.2)',  # Couleur verte semi-transparente
    line=dict(width=0),
    name='Intervalle Cancel',
    showlegend=False
))


fig.add_trace(go.Scatter(x=taille_cancel,y=cancel,mode='lines',name='Cancel',showlegend=True,line=dict(color='green')))

# fig.add_trace(go.Scatter(
#     x=taille_trades ,
#     y=trades + taille_trades_intervalle,  # Limite supérieure
#     mode='lines',
#     line=dict(width=0),
#     showlegend=False
# ))
# fig.add_trace(go.Scatter(
#     x=taille_trades ,
#     y=trades - taille_trades_intervalle,  # Limite inférieure
#     mode='lines',
#     fill='tonexty',
#     fillcolor='rgba(255, 0, 0, 0.2)',  # Couleur rouge semi-transparente
#     line=dict(width=0),
#     name='Intervalle Trades',
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(x=taille_trades,y=trades,mode='lines',name='Trades',showlegend=True,line=dict(color='red')))

#fig.update_layout(title="Intensité par Queue size avec intervalles d'incertitude",xaxis_title='Size (par Mean Event Size)',yaxis_title='Intensity (num par sec)',showlegend=True)
fig.update_layout(
    yaxis=dict(
        title='Intensity (num par sec)',
        title_font=dict(size=40, family='Times New Roman'),
        tickfont=dict(size=36, family='Times New Roman'),
        showgrid=True, gridwidth=0.5, gridcolor='lightgrey',  # Grille légère
        zeroline=False, linecolor='black'
    ),
    xaxis=dict(
        title='Imbalance',
        title_font=dict(size=40, family='Times New Roman'),
        tickfont=dict(size=36, family='Times New Roman'),
        showgrid=False, gridwidth=0.5, gridcolor='lightgrey',  # Grille légère
        zeroline=False, linecolor='black'
    ),
    plot_bgcolor='white',  # Fond blanc
    showlegend=False,
    legend=dict(
        font=dict(size=10, family='Times New Roman'),
        bordercolor='black', borderwidth=0.5
    ),
    width=2000,  # Largeur du graphe en pixels
    height=1200 
    
)

fig.show()

In [ ]:
Add = [0]
Cancel = [0]
Trade = [0]
sizes_add = [0]
sizes_cancel = [0]
sizes_trade = [0]
threshold = 100

#dic = remove_nan_from_dico(dic)
average_sizes = compute_means(dic)
intensities = dict(sorted(dic.items()))
intensities = filtrage(intensities, 50, threshold=100)

threshold_trade = 100
threshold = 1000

quarter_add = [0]
quarter_cancel = [0]
quarter_trade = [0]

for i in intensities:
    tab = np.concatenate((intensities[i][0], intensities[i][1], intensities[i][2]))
    if (len(intensities[i][0])>threshold):
            Add.append(1/np.mean(tab)*len(intensities[i][0])/len(tab))
            quarter_add.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][0])/len(tab)*1/np.sqrt(len(tab)))
            sizes_add.append(i)
    if len(intensities[i][1])!=0:
        if (len(intensities[i][1])>threshold):
            Cancel.append(1/np.mean(tab)*len(intensities[i][1])/len(tab))
            quarter_cancel.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][1])/len(tab)*1/np.sqrt(len(tab)))
            sizes_cancel.append(i)
    if len(intensities[i][2])!=0:
        if (len(intensities[i][2])>threshold_trade):
            Trade.append(1/np.mean(tab)*len(intensities[i][2])/len(tab))
            quarter_trade.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][2])/len(tab)*1/np.sqrt(len(tab)))
            sizes_trade.append(i)

fig = go.Figure()

fig.add_trace(go.Scatter(x=sizes_add/np.mean(average_event_size),y=Add,mode='lines',name='Add',showlegend=True,line=dict(color='blue')))
fig.add_trace(go.Scatter(x=sizes_add/np.mean(average_event_size),y=np.array(Add)+1.96*np.array(quarter_add),mode='lines',name='Add Upper CI',line=dict(width=0),fill=None,showlegend=False))
fig.add_trace(go.Scatter(x=sizes_add/np.mean(average_event_size),y=np.array(Add)-1.96*np.array(quarter_add),mode='lines',name='Add Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 0, 255, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_cancel/np.mean(average_event_size),y=Cancel,mode='lines',name='Cancel',showlegend=True,line=dict(color='green')))
fig.add_trace(go.Scatter(x=sizes_cancel/np.mean(average_event_size),y=np.array(Cancel)+1.96*np.array(quarter_cancel),mode='lines',name='Cancel Upper CI',line=dict(width=0),fill=None,showlegend=False))
fig.add_trace(go.Scatter(x=sizes_cancel/np.mean(average_event_size),y=np.array(Cancel)-1.96*np.array(quarter_cancel),mode='lines',name='Cancel Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 255, 0, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_trade/np.mean(average_event_size),y=Trade,mode='lines',name='Trade',showlegend=True,line=dict(color='red')))
fig.add_trace(go.Scatter(x=sizes_trade/np.mean(average_event_size),y=np.array(Trade)+1.96*np.array(quarter_trade),mode='lines',name='Trade Upper CI',line=dict(width=0),fill=None,showlegend=False))
fig.add_trace(go.Scatter(x=sizes_trade/np.mean(average_event_size),y=np.array(Trade)-1.96*np.array(quarter_trade),mode='lines',name='Trade Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(255, 0, 0, 0.2)',showlegend=False))

fig.update_layout(title="Intensité par Queue size avec intervalles d'incertitude",xaxis_title='Size (par Mean Event Size)',yaxis_title='Intensity (num par sec)',showlegend=True)
fig.show()

In [ ]:
dic = {}

average_event_size = []
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    #df['time_diff'] = df['time_diff']*1.0 # pour le modif en float c'est un timedelta là
    sizes = np.unique(np.array((np.unique(df['bid_sz_00'].to_numpy())).tolist() + (np.unique(df['ask_sz_00'].to_numpy())).tolist()))
    sizes.sort()
    dic = dico_queue_size(sizes, dic)  # Add, Cancel, Trade
    
    for row in df.itertuples():
        average_event_size.append(row.size)
        if row.side == 'A':
            taille = row.ask_sz_00
        elif row.side == 'B':
            taille = row.bid_sz_00
        if row.action == 'A':
            dic[taille][0].append(row.imbalance)
        elif row.action == 'C':
            dic[taille][1].append(row.imbalance)
        elif row.action == 'T':
            dic[taille][2].append(row.imbalance)

In [ ]:
Add = []
Cancel = []
Trade = []
sizes_add = []
sizes_cancel = []
sizes_trade = []

#dic = remove_nan_from_dico(dic)
average_sizes = compute_means(dic)
intensities = dict(sorted(dic.items()))
intensities = filtrage(intensities, 30, threshold=100)

threshold_trade = 1000
threshold = 10

quarter_add = []
quarter_cancel = []
quarter_trade = []

for i in intensities:
    tab = np.concatenate((intensities[i][0], intensities[i][1], intensities[i][2]))
    if (len(intensities[i][0])>threshold):
            Add.append(np.mean(tab)*len(intensities[i][0])/len(tab))
            quarter_add.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][0])/len(tab)*1/np.sqrt(len(tab)))
            sizes_add.append(i)
    if len(intensities[i][1])!=0:
        if (len(intensities[i][1])>threshold):
            Cancel.append(np.mean(tab)*len(intensities[i][1])/len(tab))
            quarter_cancel.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][1])/len(tab)*1/np.sqrt(len(tab)))
            sizes_cancel.append(i)
    if len(intensities[i][2])!=0:
        if (len(intensities[i][2])>threshold_trade):
            Trade.append(np.mean(tab)*len(intensities[i][2])/len(tab))
            quarter_trade.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][2])/len(tab)*1/np.sqrt(len(tab)))
            sizes_trade.append(i)

fig = go.Figure()

fig.add_trace(go.Scatter(x=sizes_add/np.mean(average_event_size),y=Add,mode='lines',name='Add',showlegend=True,line=dict(color='blue')))
#fig.add_trace(go.Scatter(y=sizes_add/np.mean(average_event_size),x=np.array(Add)+1.96*np.array(quarter_add),mode='lines',name='Add Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_add/np.mean(average_event_size),x=np.array(Add)-1.96*np.array(quarter_add),mode='lines',name='Add Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 0, 255, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_cancel/np.mean(average_event_size),y=Cancel,mode='lines',name='Cancel',showlegend=True,line=dict(color='green')))
#fig.add_trace(go.Scatter(y=sizes_cancel/np.mean(average_event_size),x=np.array(Cancel)+1.96*np.array(quarter_cancel),mode='lines',name='Cancel Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_cancel/np.mean(average_event_size),x=np.array(Cancel)-1.96*np.array(quarter_cancel),mode='lines',name='Cancel Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 255, 0, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_trade/np.mean(average_event_size),y=Trade,mode='lines',name='Trade',showlegend=True,line=dict(color='red')))
#fig.add_trace(go.Scatter(y=sizes_trade/np.mean(average_event_size),x=np.array(Trade)+1.96*np.array(quarter_trade),mode='lines',name='Trade Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_trade/np.mean(average_event_size),x=np.array(Trade)-1.96*np.array(quarter_trade),mode='lines',name='Trade Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(255, 0, 0, 0.2)',showlegend=False))

fig.update_layout(title="Queue size avec intervalles d'incertitude en fonction de l'imbalance",xaxis_title='imbalance',yaxis_title='Size (par Mean Event Size)',showlegend=True)
fig.show()

In [ ]:
queue_size_same_trade = np.array([])
queue_size_opposite_trade = np.array([])
time_trade = np.array([])

queue_size_same_cancel = np.array([])
queue_size_opposite_cancel = np.array([])
time_cancel = np.array([])

queue_size_same_add = np.array([])
queue_size_opposite_add = np.array([])
time_add = np.array([])

limite = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    df = df[:-100]
    df_cancel = df[df['action'] == 'C']
    df_cancel['ts_event'] = pd.to_datetime(df_cancel['ts_event'], errors='coerce')
    df_cancel['seconds_since_start_of_day'] = (
        df_cancel['ts_event'].dt.hour*3600+
        df_cancel['ts_event'].dt.minute*60+
        df_cancel['ts_event'].dt.second+
        df_cancel['ts_event'].dt.microsecond/1e6
    )
    df_add = df[df['action'] == 'A']
    df_add['ts_event'] = pd.to_datetime(df_add['ts_event'], errors='coerce')
    df_add['seconds_since_start_of_day'] = (
        df_add['ts_event'].dt.hour*3600+
        df_add['ts_event'].dt.minute*60+
        df_add['ts_event'].dt.second+
        df_add['ts_event'].dt.microsecond/1e6
    )
    df_trade = df[df['action'] == 'T']
    df_trade['ts_event'] = pd.to_datetime(df_trade['ts_event'], errors='coerce')
    df_trade['seconds_since_start_of_day'] = (
        df_trade['ts_event'].dt.hour*3600+
        df_trade['ts_event'].dt.minute*60+
        df_trade['ts_event'].dt.second+
        df_trade['ts_event'].dt.microsecond/1e6
    )

    queue_size_same_trade = np.concatenate([queue_size_same_trade, df_trade['size_same'].to_numpy()])
    queue_size_opposite_trade = np.concatenate([queue_size_opposite_trade, df_trade['size_opposite'].to_numpy()])
    time_trade = np.concatenate([time_trade, df_trade['seconds_since_start_of_day'].dropna().to_numpy()])
    
    queue_size_same_add = np.concatenate([queue_size_same_add, df_add['size_same'].to_numpy()])
    queue_size_opposite_add = np.concatenate([queue_size_opposite_add, df_add['size_opposite'].to_numpy()])
    time_add = np.concatenate([time_add, df_add['seconds_since_start_of_day'].dropna().to_numpy()])
    
    queue_size_same_cancel = np.concatenate([queue_size_same_cancel, df_cancel['size_same'].to_numpy()])
    queue_size_opposite_cancel = np.concatenate([queue_size_opposite_cancel, df_cancel['size_opposite'].to_numpy()])
    time_cancel = np.concatenate([time_cancel, df_cancel['seconds_since_start_of_day'].dropna().to_numpy()])

time_add = np.array(time_add, dtype=float)
time_trade = np.array(time_trade, dtype=float)
time_cancel = np.array(time_cancel, dtype=float)
print(f"A quel point c'est long: {len(time_add)}")

In [ ]:
indices_trie_add = np.argsort(time_add)


time_add = time_add[indices_trie_add]
queue_size_same_add = queue_size_same_add[indices_trie_add]
queue_size_opposite_add = queue_size_opposite_add[indices_trie_add]
group_size_add = 10000

time_trie_groups_add = [time_add[i:i + group_size_add] for i in range(0, len(time_add), group_size_add)]
queue_size_same_trie_groups_add = [queue_size_same_add[i:i + group_size_add] for i in range(0, len(queue_size_same_add), group_size_add)]
queue_size_opposite_trie_groups_add = [queue_size_opposite_add[i:i + group_size_add] for i in range(0, len(queue_size_opposite_add), group_size_add)]

time_trie_means_add = np.array([np.mean(group) for group in time_trie_groups_add])
queue_size_same_trie_means_add = np.array([np.mean(group) for group in queue_size_same_trie_groups_add])
queue_size_opposite_trie_mean_add = np.array([np.mean(group) for group in queue_size_opposite_trie_groups_add])

#imb_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in price_trie_groups])
time_trie_means_add = pd.to_datetime(time_trie_means_add, unit='s')



indices_trie_trade = np.argsort(time_trade)


time_trade = time_trade[indices_trie_trade]
queue_size_same_trade = queue_size_same_trade[indices_trie_trade]
queue_size_opposite_trade = queue_size_opposite_trade[indices_trie_trade]
group_size_trade = 700

time_trie_groups_trade = [time_trade[i:i + group_size_trade] for i in range(0, len(time_trade), group_size_trade)]
queue_size_same_trie_groups_trade = [queue_size_same_trade[i:i + group_size_trade] for i in range(0, len(queue_size_same_trade), group_size_trade)]
queue_size_opposite_trie_groups_trade = [queue_size_opposite_trade[i:i + group_size_trade] for i in range(0, len(queue_size_opposite_trade), group_size_trade)]

time_trie_means_trade = np.array([np.mean(group) for group in time_trie_groups_trade])
queue_size_same_trie_means_trade = np.array([np.mean(group) for group in queue_size_same_trie_groups_trade])
queue_size_opposite_trie_means_trade = np.array([np.mean(group) for group in queue_size_opposite_trie_groups_trade])

#imb_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in price_trie_groups])
time_trie_means_trade = pd.to_datetime(time_trie_means_trade, unit='s')


indices_trie_cancel = np.argsort(time_cancel)


time_cancel = time_cancel[indices_trie_cancel]
queue_size_same_cancel = queue_size_same_cancel[indices_trie_cancel]
queue_size_opposite_cancel = queue_size_opposite_cancel[indices_trie_cancel]
group_size_cancel = 1000

time_trie_groups_cancel = [time_cancel[i:i + group_size] for i in range(0, len(time_cancel), group_size_cancel)]
queue_size_same_trie_groups_cancel = [queue_size_same_cancel[i:i + group_size] for i in range(0, len(queue_size_same_cancel), group_size_cancel)]
queue_size_opposite_trie_groups_cancel = [queue_size_opposite_cancel[i:i + group_size] for i in range(0, len(queue_size_opposite_cancel), group_size_cancel)]

time_trie_means_cancel = np.array([np.mean(group) for group in time_trie_groups_cancel])
queue_size_same_trie_means_cancel = np.array([np.mean(group) for group in queue_size_same_trie_groups_cancel])
queue_size_opposite_trie_means_cancel = np.array([np.mean(group) for group in queue_size_opposite_trie_groups_cancel])

#imb_trie_std = np.array([np.std(group)/np.sqrt(len(group)) for group in price_trie_groups])
time_trie_means_cancel = pd.to_datetime(time_trie_means_cancel, unit='s')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_trie_means_trade,y=queue_size_same_trie_means_trade,mode='lines',name='same',line=dict(color='blue'),showlegend=True))
fig.add_trace(go.Scatter(x=time_trie_means_trade,y=queue_size_opposite_trie_means_trade,mode='lines',name='Opposite',line=dict(color='red'),showlegend=True))
fig.update_layout(title='Queue size aux trades',xaxis_title='time',yaxis_title='Queue size',showlegend=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_means_trade,y=queue_size_same_trie_means_trade,mode='markers',name='same',line=dict(color='blue'),showlegend=True))
fig.update_layout(title='Heatmap Queue size aux trades',xaxis_title='Queue size opposite',yaxis_title='Queue size same',showlegend=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_trie_means_add,y=queue_size_same_trie_means_add,mode='lines',name='same',line=dict(color='blue'),showlegend=True))
fig.add_trace(go.Scatter(x=time_trie_means_add,y=queue_size_opposite_trie_mean_add,mode='lines',name='Opposite',line=dict(color='red'),showlegend=True))
fig.update_layout(title='Queue size aux add',xaxis_title='time',yaxis_title='Queue size',showlegend=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_mean_add,y=queue_size_same_trie_means_add,mode='markers',name='same',line=dict(color='blue'),showlegend=True))
fig.update_layout(title='Heatmap Queue size aux add',xaxis_title='Queue size opposite',yaxis_title='Queue size same',showlegend=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_trie_means_cancel,y=queue_size_same_trie_means_cancel,mode='lines',name='same',line=dict(color='blue'),showlegend=True))
fig.add_trace(go.Scatter(x=time_trie_means_cancel,y=queue_size_opposite_trie_means_cancel,mode='lines',name='Opposite',line=dict(color='red'),showlegend=True))
fig.update_layout(title='Queue size aux cancel',xaxis_title='time',yaxis_title='Queue size',showlegend=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_means_cancel,y=queue_size_same_trie_means_cancel,mode='markers',name='same',line=dict(color='blue'),showlegend=True))
fig.update_layout(title='Heatmap Queue size aux cancel',xaxis_title='Queue size opposite',yaxis_title='Queue size same',showlegend=True)
fig.show()


fig = go.Figure()
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_mean_add,y=queue_size_same_trie_means_add,mode='markers',name='add',line=dict(color='red'),showlegend=True))
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_means_cancel,y=queue_size_same_trie_means_cancel,mode='markers',name='cancel',line=dict(color='green'),showlegend=True))
fig.add_trace(go.Scatter(x=queue_size_opposite_trie_means_trade,y=queue_size_same_trie_means_trade,mode='markers',name='trade',line=dict(color='blue'),showlegend=True))
fig.update_layout(title='Heatmap Queue size',xaxis_title='Queue size opposite',yaxis_title='Queue size same',showlegend=True)
fig.show()

In [ ]:
dic = {}

average_event_size = []
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df['imbalance'] = df['imbalance'].shift()
    df['Mean_price_diff'] = df['price'].shift(-50) - df['price']
    df = df.dropna()
    #df['time_diff'] = df['time_diff']*1.0 # pour le modif en float c'est un timedelta là
    sizes = np.unique(np.array((np.unique(df['bid_sz_00'].to_numpy())).tolist() + (np.unique(df['ask_sz_00'].to_numpy())).tolist()))
    sizes.sort()
    dic = dico_queue_size(sizes, dic)  # Add, Cancel, Trade
    
    for row in df.itertuples():
        average_event_size.append(row.size)
        if row.side == 'A':
            taille = row.ask_sz_00
        elif row.side == 'B':
            taille = row.bid_sz_00
        if row.action == 'A':
            dic[taille][0].append(row.Mean_price_diff)
        elif row.action == 'C':
            dic[taille][1].append(row.Mean_price_diff)
        elif row.action == 'T':
            dic[taille][2].append(row.Mean_price_diff)

In [ ]:
Add = []
Cancel = []
Trade = []
sizes_add = []
sizes_cancel = []
sizes_trade = []

#dic = remove_nan_from_dico(dic)
average_sizes = compute_means(dic)
intensities = dict(sorted(dic.items()))
intensities = filtrage(intensities, 30, threshold=100)

threshold_trade = 100
threshold = 10

quarter_add = []
quarter_cancel = []
quarter_trade = []

for i in intensities:
    tab = np.concatenate((intensities[i][0], intensities[i][1], intensities[i][2]))
    if (len(intensities[i][0])>threshold):
            Add.append(np.mean(tab)*len(intensities[i][0])/len(tab))
            quarter_add.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][0])/len(tab)*1/np.sqrt(len(tab)))
            sizes_add.append(i)
    if len(intensities[i][1])!=0:
        if (len(intensities[i][1])>threshold):
            Cancel.append(np.mean(tab)*len(intensities[i][1])/len(tab))
            quarter_cancel.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][1])/len(tab)*1/np.sqrt(len(tab)))
            sizes_cancel.append(i)
    if len(intensities[i][2])!=0:
        if (len(intensities[i][2])>threshold_trade):
            Trade.append(np.mean(tab)*len(intensities[i][2])/len(tab))
            quarter_trade.append(np.var(tab)*1/np.mean(tab)*len(intensities[i][2])/len(tab)*1/np.sqrt(len(tab)))
            sizes_trade.append(i)

fig = go.Figure()

fig.add_trace(go.Scatter(x=sizes_add/np.mean(average_event_size),y=Add,mode='lines',name='Add',showlegend=True,line=dict(color='blue')))
#fig.add_trace(go.Scatter(y=sizes_add/np.mean(average_event_size),x=np.array(Add)+1.96*np.array(quarter_add),mode='lines',name='Add Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_add/np.mean(average_event_size),x=np.array(Add)-1.96*np.array(quarter_add),mode='lines',name='Add Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 0, 255, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_cancel/np.mean(average_event_size),y=Cancel,mode='lines',name='Cancel',showlegend=True,line=dict(color='green')))
#fig.add_trace(go.Scatter(y=sizes_cancel/np.mean(average_event_size),x=np.array(Cancel)+1.96*np.array(quarter_cancel),mode='lines',name='Cancel Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_cancel/np.mean(average_event_size),x=np.array(Cancel)-1.96*np.array(quarter_cancel),mode='lines',name='Cancel Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(0, 255, 0, 0.2)',showlegend=False))

fig.add_trace(go.Scatter(x=sizes_trade/np.mean(average_event_size),y=Trade,mode='lines',name='Trade',showlegend=True,line=dict(color='red')))
#fig.add_trace(go.Scatter(y=sizes_trade/np.mean(average_event_size),x=np.array(Trade)+1.96*np.array(quarter_trade),mode='lines',name='Trade Upper CI',line=dict(width=0),fill=None,showlegend=False))
#fig.add_trace(go.Scatter(y=sizes_trade/np.mean(average_event_size),x=np.array(Trade)-1.96*np.array(quarter_trade),mode='lines',name='Trade Lower CI',fill='tonexty',line=dict(width=0),fillcolor='rgba(255, 0, 0, 0.2)',showlegend=False))

fig.update_layout(title="Price différence en fonction des queues size",xaxis_title='Size (par Mean Event Size)',yaxis_title='price difference',showlegend=True)
fig.show()